# Tome 11, Chapter 6: Stochastic differential equations (Python companion)

Exercises first, solutions at the end. The Ornstein–Uhlenbeck process simulated exactly, the strong orders of the Euler–Maruyama and Milstein schemes measured, the weak order computed, the squared Bessel processes that do and do not reach $0$, Dyson's eigenvalues that never collide, and a computer science application: **delta hedging as an algorithm**, the Black–Scholes replication run on a computer with a finite number of trades, and how its error decreases. Requires `numpy`.

In [1]:
import math
import numpy as np
rng = np.random.default_rng(1106)
Phi = np.vectorize(lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2))))

def bs_call(s, K, r, sig, tau):
    # Black-Scholes price C(t, s) with tau = T - t > 0 (Theorem 6.9)
    dp = (np.log(s / K) + (r + sig**2 / 2) * tau) / (sig * math.sqrt(tau))
    return s * Phi(dp) - K * math.exp(-r * tau) * Phi(dp - sig * math.sqrt(tau)), Phi(dp)

## Exercises

**P6.1 (★) Ornstein–Uhlenbeck, exactly.** Proposition 6.6 gives the law of $X_{t+h}$ given $X_t$: Gaussian, mean $e^{-\theta h}X_t$, variance $\frac{\sigma^2}{2\theta}(1-e^{-2\theta h})$. Simulate $10^4$ paths with $\theta=1$, $\sigma=1$ from $x=3$ with this exact recursion, and with Euler–Maruyama, step $h=0.5$. Compare the variances at $t=10$ with the stationary value $\frac12$.

**P6.2 (★) Black–Scholes by Monte Carlo.** With $s=K=100$, $r=0.02$, $\sigma=0.2$, $T=1$, compute the price from the formula, and estimate the discounted mean payoff $e^{-rT}\mathbb E(S_T-K)^+$ when $S$ has drift $r$, and when it has drift $\mu=0.1$. Which one is the price?

**P6.3 (★★) Strong orders.** For $\mathrm dX=X\,\mathrm dt+X\,\mathrm dB$, $X_0=1$, whose solution is $e^{B_t+t/2}$, measure the root mean square error at $T=1$ of the Euler–Maruyama scheme and of Milstein's scheme (which adds $\frac12\sigma\sigma'(X)((\Delta B)^2-h)$) for $h=2^{-2},\dots,2^{-8}$, using the same Brownian path for the scheme and for the exact solution. Estimate the orders from the slopes.

**P6.4 (★★) Weak order.** For the same equation, $\mathbb EX_1=e$, and the Euler scheme gives $\mathbb EX^h_1=(1+h)^{1/h}$ exactly (why?). Show numerically that the weak error is of order $h$, and explain why a Monte Carlo estimate of the weak error needs many more samples than the strong one.

**P6.5 (★★) Which dimensions reach zero.** Simulate the squared Bessel process of dimension $\delta=1,1.5,2,3$ from $z=1$ until it leaves $]0.01,4[$, by the Euler scheme with $\sqrt{\max(Z,0)}$, and compare the probability of leaving at the bottom with the formula $\frac{s(R)-s(z)}{s(R)-s(\varepsilon)}$ of Theorem 6.12. How does it behave as $\varepsilon\to0$ for each $\delta$?

**P6.6 (★★) Dyson's eigenvalues.** Run a $40\times40$ Hermitian Brownian motion from $0$ up to $t=1$, recording the eigenvalues. Plot (or tabulate) the smallest gap over time, compare the spectrum at $t=1$ with the semicircle of radius $2\sqrt{40}$, and compare the normalised spacings in the bulk with those of $40$ independent Gaussians: which one avoids small spacings?

**P6.7 (★★★) CS application: hedging with finitely many trades.** A bank sells the call of P6.2 for $C(0,s)$ and hedges it by the strategy of Theorem 6.9, but can only rebalance $n$ times, at equally spaced dates, while the real stock has drift $\mu=0.1$. Simulate $4000$ scenarios for $n=4,16,64,256,1024$ and report the mean and standard deviation of the final profit and loss $V_T-(S_T-K)^+$. How does the risk decrease with $n$?

## Solutions

### P6.1

**How to find it.** The exact recursion is $X\leftarrow e^{-\theta h}X+\sqrt{\frac{\sigma^2}{2\theta}(1-e^{-2\theta h})}\,Z$. The Euler scheme is $X\leftarrow(1-\theta h)X+\sigma\sqrt h\,Z$, an autoregression whose stationary variance is $\frac{\sigma^2h}{1-(1-\theta h)^2}=\frac{\sigma^2}{\theta(2-\theta h)}$.

In [2]:
theta, sigma, h, t, m = 1.0, 1.0, 0.5, 10.0, 10_000
X = np.full(m, 3.0); Y = np.full(m, 3.0)
a = math.exp(-theta * h); s = math.sqrt(sigma**2 / (2 * theta) * (1 - a * a))
for _ in range(int(t / h)):
    X = a * X + s * rng.standard_normal(m)
    Y = (1 - theta * h) * Y + sigma * math.sqrt(h) * rng.standard_normal(m)
print(f"exact recursion: mean {X.mean():+.3f}, variance {X.var():.3f}")
print(f"Euler-Maruyama : mean {Y.mean():+.3f}, variance {Y.var():.3f}  (its own stationary value {sigma**2 / (theta * (2 - theta * h)):.3f})")
print(f"stationary law : variance {sigma**2 / (2 * theta):.3f}")

exact recursion: mean +0.012, variance 0.494
Euler-Maruyama : mean -0.004, variance 0.647  (its own stationary value 0.667)
stationary law : variance 0.500


The exact recursion has no discretisation error at all, whatever $h$: its variance is $\frac12$ up to Monte Carlo noise. The Euler scheme with the same coarse step settles to the wrong equilibrium, variance $\frac23$ instead of $\frac12$. For linear equations, sampling the exact transition law is always better.

### P6.2

**How to find it.** Sample $S_T=s\,e^{\sigma\sqrt TZ+(\nu-\sigma^2/2)T}$ with $\nu=r$ and $\nu=\mu$.

In [3]:
s0, K, r, sig, T, mu = 100.0, 100.0, 0.02, 0.2, 1.0, 0.1
C0, delta0 = bs_call(s0, K, r, sig, T)
print(f"formula: C(0, s) = {C0:.4f}")
Z = rng.standard_normal(10**6)
for nu in (r, mu):
    pay = math.exp(-r * T) * np.maximum(s0 * np.exp((nu - sig**2 / 2) * T + sig * math.sqrt(T) * Z) - K, 0)
    print(f"drift {nu}: discounted mean payoff {pay.mean():.4f} +- {pay.std() / 1000:.4f}")

formula: C(0, s) = 8.9160
drift 0.02: discounted mean payoff 8.9214 +- 0.0138
drift 0.1: discounted mean payoff 14.3784 +- 0.0175


With drift $r$ the Monte Carlo mean agrees with the formula within its standard error; with the real drift $\mu$ the discounted mean payoff is much larger, and it is *not* the price. The price is an expectation under the risk-neutral probability (the remark after Theorem 6.9), because the bank does not bet on the stock: it replicates the option (P6.7).

### P6.3

**How to find it.** Build the coarse increments by summing a fine path of $2^{12}$ steps, so that the exact solution $e^{B_1+1/2}$ and both schemes use the same Brownian motion. For $\sigma(x)=x$, $\sigma\sigma'(x)=x$.

In [4]:
m, N = 20_000, 2**12
dW = rng.standard_normal((m, N)) * math.sqrt(1 / N)
exact = np.exp(dW.sum(1) + 0.5)
res = []
for k in range(2, 9, 2):
    n = 2**k; h = 1 / n
    d = dW.reshape(m, n, N // n).sum(2)
    em = np.ones(m); mil = np.ones(m)
    for j in range(n):
        em = em * (1 + h + d[:, j])
        mil = mil * (1 + h + d[:, j] + 0.5 * (d[:, j]**2 - h))
    res.append((h, math.sqrt(np.mean((em - exact)**2)), math.sqrt(np.mean((mil - exact)**2))))
    print(f"h = 1/{n:<4}: Euler-Maruyama {res[-1][1]:.4f}   Milstein {res[-1][2]:.4f}")
H = np.log([x[0] for x in res])
print("slopes: Euler-Maruyama %.2f, Milstein %.2f" % (np.polyfit(H, np.log([x[1] for x in res]), 1)[0],
                                                     np.polyfit(H, np.log([x[2] for x in res]), 1)[0]))

h = 1/4   : Euler-Maruyama 2.2514   Milstein 1.6195
h = 1/16  : Euler-Maruyama 1.0932   Milstein 0.5690


h = 1/64  : Euler-Maruyama 0.4648   Milstein 0.1608


h = 1/256 : Euler-Maruyama 0.2197   Milstein 0.0387
slopes: Euler-Maruyama 0.57, Milstein 0.90


Each division of $h$ by $4$ halves the Euler error and divides the Milstein error by a factor that approaches $4$: slopes close to $\frac12$ and $1$, the strong orders of Theorem 6.15 and of Milstein's scheme. The Milstein correction removes exactly the term $\frac12((\Delta B)^2-h)$ that Exercise 6.8 identifies as the source of the $\sqrt h$ error.

### P6.4

**How to find it.** The Euler step multiplies by $1+h+\Delta B$, with $\Delta B$ independent of the past and centred, so $\mathbb EX^h_{(k+1)h}=(1+h)\mathbb EX^h_{kh}$.

In [5]:
for n in (4, 16, 64, 256, 1024):
    err = math.e - (1 + 1 / n)**n
    print(f"h = 1/{n:<5}: weak error {err:.6f}   error / h = {err * n:.4f}")

h = 1/4    : weak error 0.276876   error / h = 1.1075
h = 1/16   : weak error 0.080353   error / h = 1.2857
h = 1/64   : weak error 0.020937   error / h = 1.3400
h = 1/256  : weak error 0.005290   error / h = 1.3543
h = 1/1024 : weak error 0.001326   error / h = 1.3579


The ratio error$/h$ tends to $e/2\approx1.359$, since $(1+h)^{1/h}=e^{1-h/2+O(h^2)}$: weak order $1$, twice the strong order. To *see* a weak error of $0.005$ by simulation, the Monte Carlo standard deviation $\sqrt{\operatorname{Var}X_1/m}$ (with $\operatorname{Var}X_1=e^3-e^2\approx12.7$) must be well below $0.005$, which takes several million paths; the strong error, of order $\sqrt h$, is visible with a few thousand. In practice the bias of the scheme and the statistical error are balanced against each other.

### P6.5

**How to find it.** Run all walkers in parallel and freeze each one when it leaves the interval. The scale function is $s(z)=z^{1-\delta/2}$, or $\log z$ for $\delta=2$.

In [6]:
def besq_exit_low(delta, z, eps, R, m, dt):
    Z = np.full(m, z); out = np.zeros(m, int)   # 1: left at eps, 2: left at R
    while (out == 0).any():
        a = out == 0
        Z[a] += delta * dt + 2 * np.sqrt(np.maximum(Z[a], 0)) * rng.standard_normal(a.sum()) * math.sqrt(dt)
        out[a & (Z <= eps)] = 1; out[a & (Z >= R)] = 2
    return np.mean(out == 1)

z, eps, R, m = 1.0, 0.01, 4.0, 4000
for delta in (1.0, 1.5, 2.0, 3.0):
    p = 1 - delta / 2
    exact = math.log(R / z) / math.log(R / eps) if delta == 2 else (R**p - z**p) / (R**p - eps**p)
    est = besq_exit_low(delta, z, eps, R, m, 1e-4)
    print(f"delta = {delta}: simulated {est:.3f} +- {math.sqrt(est * (1 - est) / m):.3f}   formula {exact:.3f}")

delta = 1.0: simulated 0.529 +- 0.008   formula 0.526


delta = 1.5: simulated 0.373 +- 0.008   formula 0.377


delta = 2.0: simulated 0.226 +- 0.007   formula 0.231


delta = 3.0: simulated 0.052 +- 0.004   formula 0.053


The simulation follows the formula within about two standard errors (the Euler scheme also overshoots the barriers slightly, as in Chapter 5's P5.1). As $\varepsilon\to0$ the formula tends to $1-(z/R)^{1-\delta/2}=0.5$ and $0.29$ for $\delta=1$ and $1.5$, a positive probability of reaching $0$ before $4$; for $\delta=2$ it decays like $1/\log(1/\varepsilon)$, and for $\delta=3$ like $\sqrt\varepsilon$: these processes never reach $0$ (Theorem 6.12).

### P6.6

**How to find it.** Add a Hermitian Gaussian matrix with diagonal variance $h$ and off-diagonal $\mathbb E|g|^2=h$ at each step and diagonalise. For the spacings, take the eigenvalues in the middle half of the spectrum, divide the gaps by their mean, and compare with $40$ independent $N(0,40)$ variables treated the same way. GUE spacings are well approximated by Wigner's surmise $\frac{32}{\pi^2}s^2e^{-4s^2/\pi}$, independent points by $e^{-s}$.

In [7]:
n, steps = 40, 400
h = 1 / steps
H = np.zeros((n, n), complex)
min_gap = []
for k in range(steps):
    A = rng.standard_normal((n, n)) + 1j * rng.standard_normal((n, n))
    H = H + (A + A.conj().T) / 2 * math.sqrt(h)
    ev = np.linalg.eigvalsh(H)
    min_gap.append(np.diff(ev).min())
print("smallest gap at t = 0.01, 0.1, 0.5, 1:", [round(min_gap[i], 4) for i in (3, 39, 199, 399)])
print(f"spectrum at t = 1: from {ev.min():.2f} to {ev.max():.2f}; semicircle edge 2*sqrt(n) = {2 * math.sqrt(n):.2f}")

def spacings(points):
    x = np.sort(points); k = len(x)
    g = np.diff(x[k // 4: 3 * k // 4])
    return g / g.mean()

gue, iid = [], []
for _ in range(300):
    A = rng.standard_normal((n, n)) + 1j * rng.standard_normal((n, n))
    gue.append(spacings(np.linalg.eigvalsh((A + A.conj().T) / 2)))
    iid.append(spacings(rng.standard_normal(n) * math.sqrt(n)))
gue, iid = np.concatenate(gue), np.concatenate(iid)
for name, sp in (("eigenvalues", gue), ("independent", iid)):
    print(f"{name}: P(spacing < 0.1) = {np.mean(sp < 0.1):.4f}, P(spacing < 0.25) = {np.mean(sp < 0.25):.3f}")
wig = lambda s: 32 / math.pi**2 * s**2 * math.exp(-4 * s**2 / math.pi)
print("Wigner surmise: P(s < 0.1) = %.4f, P(s < 0.25) = %.3f; exponential: %.4f, %.3f" % (
    sum(wig(0.0005 + 0.001 * i) * 0.001 for i in range(100)), sum(wig(0.0005 + 0.001 * i) * 0.001 for i in range(250)),
    1 - math.exp(-0.1), 1 - math.exp(-0.25)))

smallest gap at t = 0.01, 0.1, 0.5, 1: [np.float64(0.0159), np.float64(0.0499), np.float64(0.196), np.float64(0.133)]
spectrum at t = 1: from -12.27 to 12.00; semicircle edge 2*sqrt(n) = 12.65
eigenvalues: P(spacing < 0.1) = 0.0016, P(spacing < 0.25) = 0.016
independent: P(spacing < 0.1) = 0.0891, P(spacing < 0.25) = 0.210
Wigner surmise: P(s < 0.1) = 0.0011, P(s < 0.25) = 0.016; exponential: 0.0952, 0.221


The smallest gap never reaches $0$ (Theorem 6.14), the eigenvalues at $t=1$ fill an interval close to $[-2\sqrt{40},2\sqrt{40}]$ (the semicircle law of Chapter 10), and small spacings are rare among eigenvalues, in proportion to $s^2$ near $0$, where independent points produce them at a constant rate: the repulsion $\frac1{\lambda_i-\lambda_k}$ in Dyson's equation is visible in a histogram. That this is the spacing law of the zeros of the zeta function is the subject of Chapter 11.

### P6.7

**How to find it.** At each rebalancing date hold $a=\Phi(d_+)$ shares and put the rest of the portfolio value in the bond; between dates, the holdings are frozen while the stock moves with drift $\mu$.

In [8]:
m = 4000
for n in (4, 16, 64, 256, 1024):
    dt = T / n
    S = np.full(m, s0); V = np.full(m, C0)
    for k in range(n):
        a = bs_call(S, K, r, sig, T - k * dt)[1]
        cash = V - a * S
        S = S * np.exp((mu - sig**2 / 2) * dt + sig * math.sqrt(dt) * rng.standard_normal(m))
        V = a * S + cash * math.exp(r * dt)
    pnl = V - np.maximum(S - K, 0)
    print(f"n = {n:<5}: profit and loss mean {pnl.mean():+.3f}, standard deviation {pnl.std():.3f}")

n = 4    : profit and loss mean -0.188, standard deviation 3.336
n = 16   : profit and loss mean -0.054, standard deviation 1.647


n = 64   : profit and loss mean -0.013, standard deviation 0.844


n = 256  : profit and loss mean +0.003, standard deviation 0.428


n = 1024 : profit and loss mean -0.008, standard deviation 0.212


The mean profit and loss stays near $0$, whatever the drift of the real stock, and its standard deviation halves each time the number of trades is multiplied by $4$: the hedging error is of order $n^{-1/2}$, the same $\sqrt h$ as in Theorem 6.15, since between two dates the portfolio misses a term $\frac12\partial_{ss}C\,((\Delta S)^2-\sigma^2S^2\Delta t)$. This rate was computed by Bertsimas, Kogan and Lo, *When is time continuous?*, Journal of Financial Economics 55 (2000), 173–204. In practice transaction costs grow with $n$, and the trade-off between them and the hedging error is a classic problem of algorithmic trading (Leland, *Option pricing and replication with transactions costs*, Journal of Finance 40, 1985).